<a href="https://colab.research.google.com/github/michalis0/MGT-502-Data-Science-and-Machine-Learning/blob/main/assignments/Part 5/Assignment_part_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### DSML investigation

You are part of the Suisse Impossible Mission Force, or SIMF for short. You need to uncover a rogue agent that is trying to steal sensitive information.

Your mission, should you choose to accept it, is to find that agent before stealing any classified information. Good luck!

# Assignment part five

More information came in that suggests that the rogue agent is tampering with the sentiment annotation system of the SIMF which analyses news documents and marks their sentiment for intelligence analysis tasks. This annotation is crucial to identify documents expressing negativity towards Switzerland and its allies.

We know that the rogue agent accessed only the documents whose negative sentiment was high, and then changed their labels to positive or neutral. We will use a Huggingface model to identify which records have been tampered with.

In [ ]:
# Install the required libraries (you need to run this cell ONLY if you are running the notebook locally)
# No need to run this cell in colab!
%%capture
!pip install datasets transformers huggingface_hub
!apt-get install git-lfs
!pip install transformers[torch]
!pip install accelerate -U
!pip install openpyxl

!pip install -q transformers
%pip install ipywidgets
%pip install --upgrade transformers huggingface_hub torch



In [ ]:
# Import required packages
from transformers import pipeline, DataCollatorWithPadding
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split

torch.cuda.is_available()

# Import standard libraries
import pandas as pd
import numpy as np
import math
import bs4 as bs
import urllib.request
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact, interact_manual

# Import for text analytics
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer


# Import metrics libraries
from sklearn.metrics import confusion_matrix, accuracy_score



# 1. Getting to know our data

In [ ]:

df = pd.read_excel('https://raw.githubusercontent.com/michalis0/MGT-502-Data-Science-and-Machine-Learning/refs/heads/main/assignments/Part%205/data/Reduced_Set_2100.xlsx')

In [ ]:
df.head(2)

# 2. Sentiment re-evaluation
The SIMF has used a model to obtain the sentiment labels (found in the `evaluation` column), but we suspect it has been modified. We will now cross-reference these labels using a sentiment analysis pipeline based on the `finiteautomata/bertweet-base-sentiment-analysis` model. This is a sentiment analysis model trained on ~40k tweets. It classifies a text as `POS` (positive), `NEU` (neutral), or `NEG` (negative) sentiment.

First, we need to initialize a sentiment analysis pipeline with the pre-trained model mentioned above, making sure to set the correct value for the `task` parameter.

**Note**: Set the `top_k` argument to `None` to retrieve the probabilities for all possible sentiment labels in the output.

_This process may take some time._

In [ ]:
# Your code here

Apply the sentiment classifier to the `title` column and assign the corresponding sentiment labels to a new column in your dataframe named `hf_evaluation`.

Make sure to convert the sentiment labels from the model by replacing them with more descriptive terms like this (so that they are compatible with the SIMF labels):
- **NEU**: neutral
- **NEG**: negative
- **POS**: positive

*Hint: Be mindful of the format of the classifier’s output.*

_Beware that applying the model on all of the rows may take some time._

In [ ]:
# Your code here


Now, display the number of unique sentiment evaluations for both the Hugging Face and SIMF models to compare the distribution of labels. Note that the SIMF model evaluations are found in the `evaluation` column.

In [ ]:
# Your code here

Finally, visualize the comparison using a heatmap of the confusion matrix of `df["evaluation"]` and `df["hf_evaluation"]` to better understand where the two models align or differ.

In [ ]:
# Your code here

**Q1. Does the SIMF sentiment classifier predict more samples to be "neutral"  compared to the Hugging Face sentiment classifier?**

# 3. Identifying the altered documents

The SIMF model values are found in the `evaluation` column, while the HuggingFace model values should be found in the `hf_evaluation`, which you added to the table in the previous step.

Display:
*   The rows/records with same sentiment for both models.
*   The number of matching values.
*   The share of matching values of the total number of values.

In [ ]:
# Your code here


**Q2. How many entries are identical between the SIMF model evaluation and the Hugging Face model evaluation?**

*Note: Provide your answer as an integer (e.g., 80).*


Remember, we are looking at document that were tempered (altered). We suspect that the rogue agent accessed only the documents whose negative sentiment was high, and then changed to positive or neutral.

Create a subset with only those entries, which appear as 'positive' or 'neutral' in the original `evaluation` column, but are predicted as having a 'negative' sentiment by the Huggingface model.

Let's name this subset `df_altered`.

In [ ]:
# Your code here


**Q3. How many entries were changed from a negative evaluation (by the Hugging Face model) to a neutral or positive evaluation (in the SIMF model)?**

*Note: Provide your answer as an integer (e.g., 45).*


# 4. Using the ChangeLog dataframe to identify the userIDs of those who edited the entries.

Now that we have the list of altered documents (in `df_altered`), we need to see which users were responsible for these edits.

By combining `df_altered` with `ChangeLog`, display the userIDs belonging to the individuals tried to mask negative sentiments.

In [ ]:
ChangeLog = pd.read_csv('https://raw.githubusercontent.com/michalis0/MGT-502-Data-Science-and-Machine-Learning/refs/heads/main/assignments/Part%205/data/ChangeLogFix.csv')

In [ ]:
display(ChangeLog.head(10))

In [ ]:
# Your code here


**Q4. Which of the following users remain suspects?**

*Note: Select among the options in the Moodle*

# 5. Identifying key information in the altered documents

In this section, we will use the **TF-IDF** (Term Frequency-Inverse Document Frequency) features to identify significant terms in the *altered documents*.

Start by creating a list of all the original texts from the `news` column in the dataframe `df`.

In [ ]:
# Your code here



Initialize the `TfidfVectorizer` with unigrams (`ngram_range=(1, 1)`) and set the `stop_words` parameter to `'english'` to exclude common English words from the analysis.


Fit the vectorizer on the entire corpus obtained from the `news` column of `df` dataframe above.

In [ ]:
# Your code here


We now want to focus solely on the **"altered documents"**.

To do this, use the fitted vectorizer above to transform the `news` column of the `df_altered` dataframe (the one that contains the documents where the HuggingFace model gave a 'negative' evaluation, but the SIMF model evaluated them as 'neutral' or 'positive').

Then, convert the resulting document-term matrix into a new dataframe called `tfidf_df` for easy visualization and analysis.

In [ ]:
# Your code here


Now, we will identify the document that stands out the most among the altered documents based on the TF-IDF values.
   
1. **Sum TF-IDF Values**: For each tampered document, calculate the sum of the TF-IDF values across all tokens. This gives an overall importance score for each document.

2. **Find the Most Significant Document**: Identify the document with the highest summed TF-IDF value, which stands out the most. Display the details of this document.

In [ ]:
# Your code here


**Q5. What is the company name of the most important altered document?**

*Note: The most important altered document means the document with the highest summed TF-IDF value.*

Now, across the altered documents, let's identify the words that stand out the most, meaning those with the highest summed TF-IDF values.

To achieve this, sum the values of each column in the altered TF-IDF dataframe, since each column represents a token. Then, sort these summed values in descending order to easily identify the top 4 words with the highest TF-IDF scores.

Once you have these top 4 words, count in how many *altered documents* each top word appeared.

In [ ]:
# Your code here


**Q6. What is the token with the highest summed TF-IDF value?**

*Note: Select among the options in the Moodle*

**Q7. In how many altered documents does the third most frequent word appear ?**

*Note: Provide your answer as an integer (e.g., 45).*

## Your investigation is progressing effectively, and the list of suspects is narrowing down.

**Don't forget to answer the quiz and submit your code on Moodle before the end of the deadline.**